# Super TikTakToe: the line-layer design in self-play on Google Colab

This trains the network **`large_lines`**: the size of large (96 channels, 8 blocks) with **line layers** added
(design D5 of `src/test_designs.py`), with exactly the settings large had. It is compared with large's own log
at the same iterations. The network is saved as `alphazero_large_lines.pt` in `Super-TikTakToe-models` in your
Google Drive; nothing of large is touched.

**Now: the continuation from iteration 100 to 160.** The first 100 iterations are done (both networks stood at
about 47% against medium). From iteration 101 the search uses 400 simulations, and from 121 the learning rate
drops to 0.0003, both exactly as in large's run. Large scored over these iterations:
**47% at 104 to 120, 51.1% at 124 to 160** (about 55% at the very end, iteration 240).

1. In the Drive folder there must be `alphazero_medium.pt`, `alphazero_large_lines.pt` and
   `alphazero_large_lines_buffer.npz` (they are, from the first 100 iterations).
2. Menu **Runtime > Change runtime type**: choose the L4.
3. Run the cells from top to bottom. The first line must read `Continuing 'large_lines' network after iteration 100`.
   Every further line is one iteration; every 4th one has the check against medium (256 games).
4. It stops by itself after iteration 160. The first lines show the real time per iteration.
5. After a disconnect run all cells again: it continues where it stopped.
6. When it has ended: **Runtime > Disconnect and delete runtime**.


In [ ]:
# Google Drive: the folder with the finished network
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Super-TikTakToe-models'
import os
os.environ['SUPER_TTT_MODELS'] = MODELS

In [ ]:
# The code from GitHub (the newest version every time)
%cd /content
!test -d Super-TikTakToe || git clone https://github.com/kasparhui/Super-TikTakToe.git
%cd /content/Super-TikTakToe
!git pull

In [ ]:
# Which graphics card did this session get?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The training. Stop with the stop button of this cell: everything is saved, and running it again continues.
!python -u src/train.py run large_lines 160

In [ ]:
# The checks of the new network next to large's at the same iterations
import pandas as pd
new = pd.read_csv(f'{MODELS}/alphazero_large_lines_log.csv')
old = pd.read_csv(f'{MODELS}/alphazero_large_log.csv')
new = new[new['vs_medium'].notna()][['iteration', 'vs_medium', 'seconds']].rename(columns={'vs_medium': 'lines', 'seconds': 'seconds_lines'})
old = old[old['vs_medium'].notna()][['iteration', 'vs_medium', 'seconds']].rename(columns={'vs_medium': 'large', 'seconds': 'seconds_large'})
new.merge(old, on='iteration')

## After the training: direct match against large and medium

The newest line network (round 160) plays the finished **large** (round 240) and **medium** (round 240), with
500 and with 2000 simulations per move, 256 games per pair and setting (128 openings, each from both sides),
no randomness. Large against medium is known already: 52.3% at 500, 50.0% at 2000.
Needs `alphazero_large.pt` and `alphazero_medium.pt` in the Drive folder. It can be stopped and run again:
games already played are skipped. Results go to `alphazero_large_lines_tournament.csv`.


In [ ]:
# The direct match (guessed: 30 to 60 minutes on the L4)
!python -u -c "import sys; sys.path.insert(0, 'src'); import test_tournament as T; T.MODEL, T.ITERATIONS, T.NEWEST, T.ALSO, T.OPEN_CHART = 'large_lines', (), True, ['large', 'medium'], False; T.run(); T.summary()"
